In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from matplotlib import cm
import glob
import os

In [ ]:
csv_files = sorted(glob.glob('data/*.csv'))
datasets = {}
for f in csv_files:
    name = os.path.splitext(os.path.basename(f))[0]
    datasets[name] = pd.read_csv(f)
    print(f'Loaded {name}: {datasets[name].shape}')
print(f'\n{len(datasets)} dataset(s) loaded')

In [ ]:
def plot_stacked_waves(df, title_suffix=''):
    periods = df['period'].values
    cve_columns = [c for c in df.columns if c != 'period']
    cve_short = [c.split(' ')[0] for c in cve_columns]
    n_cves = len(cve_columns)
    n_periods = len(periods)

    data = df[cve_columns].values.astype(float).T

    interp_factor = 8
    x_raw = np.arange(n_periods)
    x_fine = np.linspace(0, n_periods - 1, (n_periods - 1) * interp_factor + 1)

    data_smooth = np.zeros((n_cves, len(x_fine)))
    for i in range(n_cves):
        data_smooth[i] = np.interp(x_fine, x_raw, data[i])

    fig = plt.figure(figsize=(20, 12))
    ax = fig.add_subplot(111, projection='3d')
    cmap = cm.get_cmap('cool', n_cves)
    y_spacing = 4

    for idx in range(n_cves - 1, -1, -1):
        y_val = idx * y_spacing
        z_vals = data_smooth[idx]

        verts = [(x_fine[0], y_val, 0)]
        for j in range(len(x_fine)):
            verts.append((x_fine[j], y_val, z_vals[j]))
        verts.append((x_fine[-1], y_val, 0))

        color = cmap(idx / max(n_cves - 1, 1))
        dark = (color[0] * 0.5, color[1] * 0.5, color[2] * 0.5, 1.0)

        poly = Poly3DCollection([verts], alpha=0.75, zorder=-idx)
        poly.set_facecolor((*color[:3], 0.6))
        poly.set_edgecolor(dark)
        poly.set_linewidth(0.4)
        ax.add_collection3d(poly)

        ax.plot(x_fine, [y_val] * len(x_fine), z_vals,
                color=dark, linewidth=1.2, zorder=-idx + 1)

    x_ticks, x_labels = [], []
    for i, p in enumerate(periods):
        if p.endswith('-Jan'):
            x_ticks.append(i)
            x_labels.append(p.split('-')[0])
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_labels, fontsize=8)
    ax.set_xlabel('Year', fontsize=10, labelpad=10)
    ax.set_xlim(n_periods - 1, 0)

    ax.set_yticks([i * y_spacing for i in range(n_cves)])
    ax.set_yticklabels(cve_short, fontsize=6, family='monospace')
    ax.set_ylabel('CWE', fontsize=10, labelpad=12)
    ax.set_ylim((n_cves - 1) * y_spacing + 1, -1)

    ax.set_zlim(0, int(data.max() * 1.15))
    ax.set_zlabel('Occurrences', fontsize=10, labelpad=10)

    ax.view_init(elev=28, azim=-50)
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('lightgray')
    ax.yaxis.pane.set_edgecolor('lightgray')
    ax.zaxis.pane.set_edgecolor('lightgray')
    ax.grid(True, alpha=0.3)

    ax.set_title(f'Stacked Waves — CWE Occurrences{title_suffix}', fontsize=14, pad=20)
    plt.tight_layout()
    plt.show()


def plot_waterfall(df, title_suffix=''):
    periods = df['period'].values
    cve_columns = [c for c in df.columns if c != 'period']
    cve_short = [c.split(' ')[0] for c in cve_columns]
    n_cves = len(cve_columns)
    n_periods = len(periods)

    heatmap_data = df[cve_columns].values.astype(float)

    upsample = 4
    n_rows_smooth = (n_periods - 1) * upsample + 1
    heatmap_smooth = np.zeros((n_rows_smooth, n_cves))
    rows_orig = np.arange(n_periods)
    rows_fine = np.linspace(0, n_periods - 1, n_rows_smooth)
    for col in range(n_cves):
        heatmap_smooth[:, col] = np.interp(rows_fine, rows_orig, heatmap_data[:, col])

    fig, ax = plt.subplots(figsize=(16, 10))
    x_edges = np.arange(n_cves + 1) - 0.5
    y_edges = np.linspace(-0.5, n_rows_smooth - 0.5, n_rows_smooth + 1)

    im = ax.pcolormesh(x_edges, y_edges, heatmap_smooth, cmap='inferno', shading='flat')
    ax.set_ylim(n_rows_smooth - 0.5, -0.5)
    ax.set_xlim(-0.5, n_cves - 0.5)

    ax.set_xticks(np.arange(n_cves))
    ax.set_xticklabels(cve_short, fontsize=7, rotation=45, ha='right', family='monospace')
    ax.set_xlabel('CWE', fontsize=11)

    y_ticks_orig, y_labels_list = [], []
    for i, p in enumerate(periods):
        if p.endswith('-Jan'):
            y_ticks_orig.append(i)
            y_labels_list.append(p.split('-')[0])

    y_ticks_smooth = [int(t * upsample) for t in y_ticks_orig]
    ax.set_yticks(y_ticks_smooth)
    ax.set_yticklabels(y_labels_list, fontsize=9)
    ax.set_ylabel('Year', fontsize=11)

    cbar = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.03)
    cbar.set_label('Occurrences', fontsize=10)

    ax.set_title(f'Waterfall (Radio-Signal Style){title_suffix}', fontsize=14, pad=12)
    plt.tight_layout()
    plt.show()


def plot_cwe_grid(df, title_suffix=''):
    periods = df['period'].values
    cve_columns = [c for c in df.columns if c != 'period']
    cve_short = [c.split(' ')[0] for c in cve_columns]
    n_cves = len(cve_columns)

    cols = 5
    rows = int(np.ceil(n_cves / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(22, rows * 3), sharex=True)
    axes = axes.flatten()

    x = np.arange(len(periods))
    year_ticks, year_labels = [], []
    for i, p in enumerate(periods):
        if p.endswith('-Jan'):
            year_ticks.append(i)
            year_labels.append(p.split('-')[0])

    cmap = cm.get_cmap('cool', n_cves)

    for i, col_name in enumerate(cve_columns):
        ax = axes[i]
        color = cmap(i / max(n_cves - 1, 1))
        vals = df[col_name].values.astype(float)

        ax.fill_between(x, vals, alpha=0.3, color=color)
        ax.plot(x, vals, color=color, linewidth=1.2)

        ax.set_title(cve_short[i], fontsize=9, fontweight='bold', pad=4)
        ax.set_xticks(year_ticks)
        ax.set_xticklabels(year_labels, fontsize=7, rotation=45)
        ax.tick_params(axis='y', labelsize=7)
        ax.set_xlim(0, len(periods) - 1)
        ax.grid(True, alpha=0.2)

    for j in range(n_cves, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f'CWE Occurrences Over Time{title_suffix}', fontsize=15, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
for name, df in datasets.items():
    suffix = f'  [{name}]'
    print(f'\n{"="*60}\n  {name}\n{"="*60}')
    plot_stacked_waves(df, suffix)
    plot_waterfall(df, suffix)
    plot_cwe_grid(df, suffix)